In [0]:
%pip install yfinance

In [0]:
%restart_python

In [0]:
import yfinance as yf
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

tickers = ["ITUB4.SA", "WEGE3.SA", "HMC", "EURUSD=X"]

dados_lista = []
for ticker in tickers:
    print(f"A descarregar dados para: {ticker}")
    df_temp = yf.download(ticker, period="2y").reset_index()
    
    if isinstance(df_temp.columns, pd.MultiIndex):
        df_temp.columns = [col[0] for col in df_temp.columns]
        
    df_temp['Ticker'] = ticker
    dados_lista.append(df_temp)

df_consolidado = pd.concat(dados_lista, ignore_index=True)

df_consolidado.columns = [str(col).replace(" ", "_") for col in df_consolidado.columns]

spark_df = spark.createDataFrame(df_consolidado)

spark_df = spark_df.withColumn("data_ingestao_bronze", current_timestamp())

display(spark_df)

table_name = "dados_financeiros_bronze"

spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

print(f"Sucesso! Dados guardados na camada Bronze como tabela: {table_name}")

In [0]:
from pyspark.sql.functions import col, to_date, round, current_timestamp

df_bronze = spark.table("dados_financeiros_bronze")

df_silver = (df_bronze
    .withColumnRenamed("Date", "data")
    .withColumnRenamed("Open", "abertura")
    .withColumnRenamed("High", "maxima")
    .withColumnRenamed("Low", "minima")
    .withColumnRenamed("Close", "fechamento")
    .withColumnRenamed("Volume", "volume")
    .withColumnRenamed("Ticker", "ticker")

    .withColumn("data", to_date(col("data")))

    .withColumn("abertura", round(col("abertura"), 2))
    .withColumn("maxima", round(col("maxima"), 2))
    .withColumn("minima", round(col("minima"), 2))
    .withColumn("fechamento", round(col("fechamento"), 2))

    .filter(col("volume") > 0)
    .filter(col("fechamento").isNotNull())
    
    .dropDuplicates(["data", "ticker"])

    .withColumn("data_processamento_silver", current_timestamp())

    .select("data", "ticker", "abertura", "maxima", "minima", "fechamento", "volume", "data_processamento_silver")
)

display(df_silver)

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dados_financeiros_silver")

print("Sucesso! Dados limpos, padronizados e salvos na camada Silver.")

In [0]:
%sql
CREATE OR REPLACE TABLE dados_financeiros_gold AS

WITH cotacoes_diarias AS (
    SELECT 
        data,
        ticker,
        fechamento
    FROM dados_financeiros_silver
),

metricas_avancadas AS (
    SELECT
        data,
        ticker,
        fechamento,
        
        AVG(fechamento) OVER (
            PARTITION BY ticker 
            ORDER BY data 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS media_movel_7d,
        
        LAG(fechamento, 1) OVER (
            PARTITION BY ticker 
            ORDER BY data
        ) AS fechamento_anterior
        
    FROM cotacoes_diarias
)

SELECT 
    data,
    ticker,
    fechamento,
    ROUND(media_movel_7d, 2) AS media_movel_7d,
    fechamento_anterior,

    ROUND(((fechamento - fechamento_anterior) / fechamento_anterior) * 100, 2) AS variacao_percentual_diaria
FROM metricas_avancadas
WHERE fechamento_anterior IS NOT NULL;

In [0]:
%sql
OPTIMIZE dados_financeiros_gold ZORDER BY (ticker, data);

In [0]:
%sql
SELECT * FROM dados_financeiros_gold
ORDER BY ticker, data DESC
LIMIT 20;